<!-- nav -->
[← 7 · One rule, many languages](07_One_Rule_Many_Languages.ipynb) · [Home](../../README.md) · [Expressions design →](../../docs/EXPRESSIONS.md)

# 8 · Rules for firmware and hardware

**The problem.** The logger's firmware, in C, checks each packet before sending it, and its FPGA's verification
engineers check the same condition with SystemVerilog assertions. In C, `uint8_t + uint8_t` is an `int`, and signed
overflow is undefined. In SystemVerilog, a bit can be `x`, unknown, or `z`, undriven. A rule that is right in
TypeScript can be wrong in either.

The Ccpp and SystemVerilog dialects model those languages' rules exactly, so you can translate the packet rule of case
study 4 and evaluate it the way the firmware and the hardware will.

In [1]:
import { Domains as D, Evaluators, Expressions as E } from "@mbse/expressions";
import * as Ccpp from "@mbse/expressions/Dialects/Ccpp";
import * as SystemVerilog from "@mbse/expressions/Dialects/SystemVerilog";
import * as Translators from "@mbse/expressions/Translators";

const int16 = new D.OfInteger.Builder().width(16n).create();
const uint8 = new D.OfInteger.Builder().width(8n).signed(false).create();
const [r, s] = [E.variable("raw"), E.variable("status")];
const packet_ok = r.ge(E.literal(200n, int16)).and_(r.le(E.literal(800n, int16)))
  .and_(s.bitand(E.literal(0b100n, uint8)).eq(E.literal(0n, uint8))).data;
const to_c = Translators.between(E.DIALECT, Ccpp.Expressions.DIALECT);
const to_sv = Translators.between(E.DIALECT, SystemVerilog.Expressions.DIALECT);

## Step 1: C

The translation writes typed literals as C writes them, with casts to the `<stdint.h>` types:

In [2]:
const c = to_c.forward(packet_ok);
console.log(Ccpp.Text.ToText(c));
for (const status of [0n, 0b100n]) {
  console.log(String(status), Ccpp.Evaluators.OfAny(c, { raw: new D.Value(int16, 523n), status: new D.Value(uint8, status) }));
}

raw >= (int16_t)200 && raw <= (int16_t)800 && (status & (uint8_t)4) == (uint8_t)0


0 true


4 false


## Step 2: where C differs

C promotes small integers to `int` before arithmetic. Basic has no implied promotions (case study 4), so the same sum
overflows a `uint8` in Basic and is simply 300 in C:

In [3]:
const total = E.literal(200n, uint8).add(E.literal(100n, uint8)).data;
console.log(Ccpp.Text.ToText(to_c.forward(total)), "=", String(Ccpp.Evaluators.OfAny(to_c.forward(total), {})));
try {
  Evaluators.OfAny(total);
} catch (error) {
  console.log("Basic:", (error as Error).message);
}

(uint8_t)200 + (uint8_t)100 = 300


Basic: add overflows uint8: 300


Signed overflow is undefined behavior in C: a compiler may assume it never happens. The evaluator throws instead of
picking a result, which is what a test should do:

In [4]:
try {
  Ccpp.Evaluators.OfAny(Ccpp.Expressions.binary("+", Ccpp.Expressions.identifier("x"), 1n), { x: 2147483647n });
} catch (error) {
  console.log((error as Error).message);
}

signed overflow in +: 2147483648 does not fit 'int'


## Step 3: SystemVerilog

SystemVerilog writes sized literals: `16'sd200` is a signed 16-bit decimal. Its evaluator gives a 4-state value,
written as SystemVerilog would display it:

In [5]:
const sv = to_sv.forward(packet_ok);
console.log(SystemVerilog.Text.ToText(sv));
console.log(String(SystemVerilog.Evaluators.OfAny(sv, { raw: new D.Value(int16, 523n), status: new D.Value(uint8, 0n) })));

raw >= 16'sd200 && raw <= 16'sd800 && (status & 8'd4) == 8'd0


1'b1


## Step 4: x and z

In simulation, a register that was never reset holds `x`. Whether that matters depends on the bit: the mask
looks only at bit 2, so an `x` elsewhere doesn't change the answer, but an `x` in bit 2 makes the whole condition `x`,
which an assertion reports as a failure to know. `Logic.of` makes a 4-state value from its bits:

In [6]:
const Logic = SystemVerilog.Domains.Logic;
for (const bits of ["00000000", "00000100", "000000x0", "00000x00"]) {
  console.log(bits, String(SystemVerilog.Evaluators.OfAny(sv, { raw: new D.Value(int16, 523n), status: Logic.of(bits) })));
}

00000000 1'b1


00000100 1'b0


000000x0 1'b1


00000x00 1'bx


This is Kleene's unknown again, in hardware's terms: `x` where the data doesn't decide.

## Step 5: what has no counterpart

`has` asks whether a property is present. A C struct or a SystemVerilog register always has its fields, so there's
nothing to translate it to, and the translator refuses, naming the term:

In [7]:
for (const translator of [to_c, to_sv]) {
  try {
    translator.forward(E.variable("this").has("celsius"));
  } catch (error) {
    console.log((error as Error).message);
  }
}

Basic operation 'has' has no Ccpp counterpart


Basic operation 'has' has no SystemVerilog counterpart


## Why it works this way

- **Model the language, don't approximate it.** Firmware and hardware bugs live in exactly the corners where languages
  differ: promotions, wrapping, undefined behavior, `x`. An evaluator that follows C's or IEEE 1800's rules finds them
  in a test, before the hardware does.
- **The neutral rule stays neutral.** Basic doesn't take C's promotions or SystemVerilog's sizing. The translator
  writes them out, and the target's evaluator applies them.
- **Refuse what can't be said.** A rule about whether data is present has no meaning for a fixed-layout packet; a
  silent approximation would be a bug in the translation.

That's the series: a rule written once, as data, checked, stored, analyzed, specialized, and run in each language its
users work in.

<!-- nav -->
[← 7 · One rule, many languages](07_One_Rule_Many_Languages.ipynb) · [Home](../../README.md) · [Expressions design →](../../docs/EXPRESSIONS.md)